In [22]:
import os
import hashlib
import zlib
import uuid

class VCS:
    def __init__(self, root_dir='.pygit'):
        self.root_dir = root_dir
        self.objects_dir = os.path.join(root_dir, "objects")

        os.makedirs(self.objects_dir, exist_ok=True)

    def store_blob(self, data: bytes) -> str:
        """
        Creates a file with the name that is generated after
        hashing the data.
        This data is stored in this file.
        If the hash already exists, it does not create a new one,
        else creates a new file with the generated hash.
        """
        object_hash = hashlib.sha1(data).hexdigest()
        path = os.path.join(self.objects_dir, object_hash)

        if not os.path.exists(path):
            compressed = zlib.compress(data)
            with open(path, "wb") as f:
                f.write(compressed)
        return object_hash

    def retrieve_blob(self, object_hash: str) -> bytes:
        """
        Return the contents of the file if it exists
        """
        if self.verify_blob(object_hash):
            path = os.path.join(self.objects_dir, object_hash)
            with open(path, 'rb') as f:
                return zlib.decompress(f.read())
        raise ValueError("The data seems to be corrupted.")

    def verify_blob(self, object_hash: str) -> bool:
        """
        Checks if the data has been corrupted by recalculating the
        hash of the data present in the file and comparing it with
        the file name
        """
        path = os.path.join(self.objects_dir, object_hash)
        if not os.path.exists(path):
            raise FileNotFoundError(f"The object you're requesting does not exists.")

        chunk_size = 8192
        decompressor = zlib.decompressobj()
        with open(path, 'rb') as f:
            while True:
                data_chunk = f.read(chunk_size)
                print(decompressor.decompress(data_chunk))

        return object_hash == hashlib.sha1(data).hexdigest()

    def store_file(self, file_path: str) -> str:
        """
        Generates the hash and stores the files
        """
        # if not os.path.exists(file_path):
        #     raise FileNotFoundError(f"The file does not exists.")

        # chunk_size = 8000
        # hasher = hashlib.sha1()
        # with open(file_path, 'rb') as f:
        #     while True:
        #         data = f.read(chunk_size)
        #         if not data:
        #             break
        #         hasher.update(data)
        
        # object_hash = hasher.hexdigest()

        # path = os.path.join(self.objects_dir, object_hash)
        # if not os.path.exists(path):
        #     compressor = zlib.compressobj()
        #     with open(file_path, 'rb') as f_r:
        #         with open(path, 'wb') as f_d:
        #             while True:
        #                 data = f_r.read(chunk_size)
        #                 if not data:
        #                     break
        #                 compressed_chunk = compressor.compress(data)
        #                 f_d.write(compressed_chunk)
        #             f_d.write(compressor.flush())

        # return object_hash

        if not os.path.exists(file_path):
            raise FileNotFoundError("The mentioned file does not exists.")
        
        chunk_size = 8192
        hasher = hashlib.sha1()
        compressor = zlib.compressobj()
        temp_file_path = os.path.join(self.objects_dir, f"tmp_{uuid.uuid4()}")


        with open(file_path, 'rb') as f_in:
            with open(temp_file_path, 'wb') as f_out:
                while True:
                    data_chunk = f_in.read(chunk_size)
                    if not data_chunk:
                        break
                    hasher.update(data_chunk)
                    f_out.write(compressor.compress(data_chunk))
        
        with open(temp_file_path, 'wb') as f:
            f.write(compressor.flush())
        
        file_hash = hasher.hexdigest()

        blob_path = os.path.join(self.objects_dir, file_hash)

        if os.path.exists(blob_path):
            os.remove(temp_file_path)
        else:
            os.rename(temp_file_path, blob_path)
        
        return file_hash

    def retrieve_file(self, object_hash: str) -> bytes:
        # if self.verify_blob(object_hash):
        path = os.path.join(self.objects_dir, object_hash)
        with open(path, 'rb') as f:
            compressed_data = f.read()
        return zlib.decompress(compressed_data)
        # raise ValueError("The data seems to be corrupted.")
                
vcs = VCS()
# hash_1 = vcs.store_blob(b'Hello')
# print(hash_1)
# print(vcs.retrieve_blob(hash_1))
h1 = vcs.store_file('test.txt')
print(h1)
print(vcs.retrieve_file(h1))

4bcf82e72c6a1bb1d8b3c2249dfec232f1d6de3d


error: Error -3 while decompressing data: incorrect header check

In [6]:
def read_in_chunks(file_path, chunk_size=8000):
    with open(file_path, 'rb') as file:
        while True:
            data = file.read(chunk_size)
            if not data:
                break
            print(data)

read_in_chunks('test.txt')

b'Hello, this is a simple test file'


In [9]:
# Test of my retention
import hashlib
import os

class VCS_p:
    def __init__(self, root_dir=".git"):
        self.root_dir = root_dir
        self.objects_dir = os.path.join(root_dir, "objects")

        os.makedirs(self.objects_dir, exist_ok=True)

    def store_blob(self, data: bytes) -> str:
        hash = hashlib.sha1(data).hexdigest()

        path = os.path.join(self.objects_dir, hash)

        if not os.path.exists(path):
            with open(path, 'wb') as f:
                f.write(data)
        return hash

    def retrieve_blob(self, hash: str) -> bytes:
        path = os.path.join(self.objects_dir, hash)

        if os.path.exists(path):
            with open(path, 'rb') as f:
                return f.read()

        raise FileNotFoundError("The requested file does not exists.")

    def hash_file(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            raise FileNotFoundError("The requested file does not exists.")
        with open(file_path, 'rb') as f:
            file_data = f.read()

        return self.store_blob(file_data)
        

v = VCS_p()
hash_1 = v.hash_file('./test.txt')
print(hash_1)
print(v.retrieve_blob(hash_1).decode('utf-8'))

4bcf82e72c6a1bb1d8b3c2249dfec232f1d6de3d
Hello, this is a simple test file


In [29]:
path = os.path.join('test.txt')
with open(path, 'rb') as f:
    data = f.read()
print(data)

compressed_data = zlib.compress(data)
# print(compressed_data)

with open('test.zlib', 'wb') as f:
    f.write(compressed_data)

with open('test.zlib', 'rb') as f:
    print(zlib.decompress(f.read()))

b'Hello, this is a simple test file'
b'Hello, this is a simple test file'


In [60]:
import hashlib
import os
import uuid
import zlib

class VersionControlSystem:
    """
    The main class implementation of the version control system.
    """
    def __init__(self, root_directory: str = '.git'):
        self.root_directory = root_directory
        self.objects_directory = os.path.join(self.root_directory, 'objects')
        self.chunk_size = 8192
        os.makedirs(self.objects_directory, exist_ok=True)

    def store_str_blob(self, str_data: str) -> str:
        data = str_data.encode('utf-8')
        compressed_data = zlib.compress(data)
        hash = hashlib.sha1(data).hexdigest()
        obj_path = os.path.join(self.objects_directory, hash)
        if not os.path.exists(obj_path):
            with open(obj_path, 'wb') as f:
                f.write(compressed_data)
        return hash
    
    def validate_str_blob(self, str_hash: str) -> bool:
        hash_path = os.path.join(self.objects_directory, str_hash)
        if not os.path.exists(hash_path):
            raise FileNotFoundError("Requested file is not present!")
        with open(hash_path, 'rb') as f:
            compressed_data = f.read()
        data = zlib.decompress(compressed_data)
        recalculated_hash = hashlib.sha1(data).hexdigest()

        return str_hash == recalculated_hash
    
    def retrieve_str_blob(self, str_hash: str) -> str:
        if self.validate_str_blob(str_hash):
            hash_path = os.path.join(self.objects_directory, str_hash)
            with open(hash_path, 'rb') as f:
                compressed_data = f.read()
            return zlib.decompress(compressed_data).decode('utf-8')
        return "File corrupted"

    def store_file_blob(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            raise FileNotFoundError("Mentioned file not found!")
        temp_file = os.path.join(self.objects_directory, f"tmp_{uuid.uuid4()}")

        hasher = hashlib.sha1()
        compressor = zlib.compressobj()

        try:
            with open(file_path, 'rb') as f_in, open(temp_file, 'wb') as f_out:
                while True:
                    data_chunk = f_in.read(self.chunk_size)
                    if not data_chunk:
                        break
                    hasher.update(data_chunk)
                    f_out.write(compressor.compress(data_chunk))
            with open(temp_file, 'ab') as f:
                f.write(compressor.flush())
            hash = hasher.hexdigest()
            obj_path = os.path.join(self.objects_directory, hash)
            if os.path.exists(obj_path):
                os.remove(temp_file)
            else:
                os.rename(temp_file, obj_path)
            return hash
        except Exception as e:
            if os.path.exists(temp_file):
                os.remove(temp_file)
            raise e
        
    def validate_file_blob(self, file_hash: str) -> bool:
        hash_path = os.path.join(self.objects_directory, file_hash)
        if not os.path.exists(hash_path):
            raise FileNotFoundError("Requested file not present.")
        decompressor = zlib.decompressobj()
        hasher = hashlib.sha1()
        with open(hash_path, 'rb') as f:
            while True:
                compressed_data_chunk = f.read(self.chunk_size)
                if not compressed_data_chunk:
                    break
                data = decompressor.decompress(compressed_data_chunk)
                hasher.update(data)
            hasher.update(decompressor.flush())
        return file_hash == hasher.hexdigest()
    
    def retrieve_file_blob(self, file_hash: str):
        decompressor = zlib.decompressobj()
        if self.validate_file_blob(file_hash):
            hash_path = os.path.join(self.objects_directory, file_hash)
            with open(hash_path, 'rb') as f:
                while True:
                    compressed_data = f.read(self.chunk_size)
                    if not compressed_data:
                        break
                    yield (decompressor.decompress(compressed_data))
                yield (decompressor.flush())
            
v = VersionControlSystem()
h1 = v.store_file_blob("test.txt")
print(f"File Hash: {h1}")
print(f"Validation Result: {v.validate_file_blob(h1)}")
for chunk in v.retrieve_file_blob(h1):
    print(chunk, end=" ")


File Hash: 907d14fb3af2b0d4f18c2d46abe8aedce17367bd
Validation Result: True
b'Hello, World' b'' 

In [54]:
def read_file(file_path: str):
    with open(file_path, 'rb') as f:
        while True:
            data = f.read(1024)
            if not data:
                break
            print(data)
read_file('test.txt')

b'Hello, World'
